In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import torch
from torch import nn
from torch.utils.data import DataLoader

from src.preprocessing import build_vocab
from src.dataset import DisasterTweetsDataset, collate_batch, collate_test_batch
from src.evaluate import evaluate
from src.utils import seed_everything
from src.models import SimpleRNNClassifier, GRUClassifier

In [2]:
df = pd.read_csv("data/train.csv")
test = pd.read_csv("data/test.csv")

In [3]:
print(torch.__version__)

2.12.1+cpu


In [4]:
device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

seed_everything(42)
device

device(type='cpu')

In [5]:
# CHECK FOR DATA IMBALANCE
target_0 = (df["target"] == 0).sum() # No disaster: Target = 0

target_1 = (df["target"] == 1).sum() # Disaster: Target = 1


print('Target = 0: ',round(target_0/len(df),2))
print('Target = 1: ',round(target_1/len(df),2))


Target = 0:  0.57
Target = 1:  0.43


In [6]:
df.head()

,id,keyword,location,text,target
0,1,NaN,NaN,Our Deeds are the Reason of this #earthquake M...,1
1,4,NaN,NaN,Forest fire near La Ronge Sask. Canada,1
2,5,NaN,NaN,All residents asked to 'shelter in place' are ...,1
3,6,NaN,NaN,"13,000 people receive #wildfires evacuation or...",1
4,7,NaN,NaN,Just got sent this photo from Ruby #Alaska as ...,1


In [7]:
# TRAIN TEST SPLIT 
train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["target"]
)

len(train_df), len(val_df)

(6090, 1523)

In [8]:
vocab = build_vocab(train_df["text"], min_freq=2, max_vocab_size=20000)

print(len(vocab))

5772


In [9]:
batch_size = 64
import numpy as np
train_dataset = DisasterTweetsDataset(train_df, vocab)
val_dataset = DisasterTweetsDataset(val_df, vocab)
test_dataset = DisasterTweetsDataset(test, vocab, is_test=True)

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    collate_fn=lambda batch: collate_batch(batch, vocab)
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    collate_fn=lambda batch: collate_batch(batch, vocab)
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    collate_fn=lambda batch: collate_test_batch(batch, vocab)
)

MODEL

In [10]:
# CREATE THE MODEL

model = SimpleRNNClassifier(
    vocab_size=len(vocab),
    embedding_dim=128,
    hidden_dim=128,
    dropout=0.3
).to(device)

model

SimpleRNNClassifier(
  (embedding): Embedding(5772, 128, padding_idx=0)
  (rnn): RNN(128, 128, batch_first=True, bidirectional=True)
  (dropout): Dropout(p=0.3, inplace=False)
  (fc): Linear(in_features=256, out_features=1, bias=True)
)

In [11]:
model = GRUClassifier(
    vocab_size=len(vocab),
    embedding_dim=128,
    hidden_dim=128,
    dropout=0.3,
    num_layers=1
).to(device)

model

GRUClassifier(
  (embedding): Embedding(5772, 128, padding_idx=0)
  (gru): GRU(128, 128, batch_first=True, bidirectional=True)
  (dropout): Dropout(p=0.3, inplace=False)
  (fc): Linear(in_features=256, out_features=1, bias=True)
)

In [12]:
model = SimpleRNNClassifier(
    vocab_size=len(vocab),
    embedding_dim=128,
    hidden_dim=128,
    dropout=0.3
).to(device)

model

SimpleRNNClassifier(
  (embedding): Embedding(5772, 128, padding_idx=0)
  (rnn): RNN(128, 128, batch_first=True, bidirectional=True)
  (dropout): Dropout(p=0.3, inplace=False)
  (fc): Linear(in_features=256, out_features=1, bias=True)
)

In [13]:
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()

    total_loss = 0

    for x, lengths, y in loader:
        x = x.to(device)
        lengths = lengths.to(device)
        y = y.to(device)

        optimizer.zero_grad()

        logits = model(x, lengths)
        loss = criterion(logits, y)

        loss.backward()

        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()

        total_loss += loss.item() * x.size(0)

    return total_loss / len(loader.dataset)

TRAIN

In [14]:
num_epochs = 10

best_f1 = 0
best_state = None

for epoch in range(num_epochs):
    train_loss = train_one_epoch(model, train_loader, optimizer, criterion)
    val_loss, val_f1, val_probs, val_labels = evaluate(
        model,
        val_loader,
        criterion,
        threshold=0.5
    )

    print(
        f"Epoch {epoch+1:02d} | "
        f"train_loss={train_loss:.4f} | "
        f"val_loss={val_loss:.4f} | "
        f"val_f1={val_f1:.4f}"
    )

    if val_f1 > best_f1:
        best_f1 = val_f1
        best_state = model.state_dict()

Epoch 01 | train_loss=0.6404 | val_loss=0.5853 | val_f1=0.5367
Epoch 02 | train_loss=0.5173 | val_loss=0.5400 | val_f1=0.6615
Epoch 03 | train_loss=0.4439 | val_loss=0.5630 | val_f1=0.6809
Epoch 04 | train_loss=0.3654 | val_loss=0.5503 | val_f1=0.7042
Epoch 05 | train_loss=0.2801 | val_loss=0.6273 | val_f1=0.6865
Epoch 06 | train_loss=0.2164 | val_loss=0.6997 | val_f1=0.6932
Epoch 07 | train_loss=0.1646 | val_loss=0.7990 | val_f1=0.7023
Epoch 08 | train_loss=0.1197 | val_loss=0.8861 | val_f1=0.7104
Epoch 09 | train_loss=0.0917 | val_loss=0.9288 | val_f1=0.6969
Epoch 10 | train_loss=0.0811 | val_loss=0.9888 | val_f1=0.7085


In [15]:
model.load_state_dict(best_state)

<All keys matched successfully>